# where-clip-negative — ex1: relu via torch.where

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `where-clip-negative`. Running the final beacon cell reports progress against the `PyTorch: where to clip negative` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: where to clip negative` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`where-clip-negative`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "where-clip-negative"
DD_SUBTOPIC = "PyTorch: where to clip negative"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## torch.where for conditional selection — quick refresher

`torch.where(cond, a, b)` returns a tensor whose entries come from `a` where `cond` is true and from `b` where it's false. All three tensors broadcast against each other.

**ReLU via where.** `torch.where(x < 0, torch.zeros_like(x), x)` is exactly ReLU — clip negatives to zero, pass positives through. The same pattern with `x > 6` and `6.0` gives ReLU6; with custom values it gives clamped activations.

**Why prefer `where` over masked-assign.** `out = x.clone(); out[x<0] = 0` works but breaks autograd in subtle ways (in-place on a clone still has detach-y semantics). `torch.where` is differentiable and composable — gradients flow through both branches based on `cond`.

### Exercise 1 — relu via torch.where

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply `torch.where(x < 0, zeros, x)` to implement ReLU as a branching selection over the input tensor.
> Keywords: where, relu, conditional, broadcasting
> ```

**KCs targeted:** `where-tertiary-op`, `where-zeros-like-pattern`

Implement `ex1_relu_via_where(x)`. ReLU is defined as `max(0, x)` elementwise. Express it as a `torch.where` call:

1. Build the condition `x < 0` (a boolean tensor with the same shape as `x`).
2. Use `torch.where(cond, zeros_like(x), x)`: where `cond` is true (i.e. `x` is negative), pick `0.0`; otherwise keep `x`.
3. Return the result.

Input: any float tensor.
Output: same shape as input; negatives are zeroed, non-negatives pass through.

You must use `torch.where` (NOT `x.clamp(min=0)` / `F.relu`).

In [ ]:
def ex1_relu_via_where(x: Tensor) -> Tensor:
    return t.where(x < 0, t.zeros_like(x), x)


<details><summary>Solution</summary>

```python
def ex1_relu_via_where(x: Tensor) -> Tensor:
    return t.where(x < 0, t.zeros_like(x), x)
```

**Argument order.** `torch.where(cond, a, b)` is 'cond ? a : b' — think C ternary. Here `cond = x < 0` returns true for negatives, so the `a` branch (`zeros_like(x)`) handles negatives and the `b` branch (`x`) handles non-negatives.

**Why `zeros_like(x)` not just `0`.** `torch.where` requires the two value branches to have a common dtype with `x`. Using `zeros_like(x)` is the safest: it matches `dtype`, `device`, and shape. A scalar `0` also works on modern PyTorch but is more fragile across dtypes.

**Generalization.** Swap the condition and constants to build any clipped activation: `where(x > 6, 6.0, x)` after ReLU = ReLU6; `where(x < lo, lo, where(x > hi, hi, x))` = hard-clamp.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()